In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, recall_score
import tensorflow as tf

In [ ]:
train_df = pd.read_csv('/content/training_data_binary.csv')
test_df = pd.read_csv('/content/testing_data_binary.csv')
X_train = train_df['prompt']
y_train = train_df['jailbreak']
X_test = test_df['prompt']
y_test = test_df['jailbreak']

In [ ]:
max_words = 5000
max_len = 100
tokenizer = Tokenizer(num_words=max_words, lower=True, split=' ')
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len)
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len)

In [ ]:
accuracies, aucs, tprs, fnrs = [], [], [], []

In [ ]:
from tqdm import tqdm

In [ ]:
for run in tqdm(range(30), desc="Running LSTM Experiments"):
    model = Sequential()
    model.add(Embedding(max_words, 100, input_length=max_len))
    model.add(SpatialDropout1D(0.2))
    model.add(Bidirectional(LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)))
    model.add(Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  metrics=['accuracy'])

    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    model.fit(X_train_pad, y_train, epochs=40, batch_size=32,
              validation_data=(X_test_pad, y_test),
              callbacks=[early_stopping], verbose=0)

    y_pred_prob = model.predict(X_test_pad).ravel()
    y_pred = (y_pred_prob >= 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_prob)
    tpr = recall_score(y_test, y_pred)  # TPR = Recall
    fnr = 1 - tpr

    accuracies.append(acc)
    aucs.append(auc)
    tprs.append(tpr)
    fnrs.append(fnr)

In [ ]:
print("\n==== Final Results ====")
print(f"Accuracy: Mean = {np.mean(accuracies):.4f}, Std = {np.std(accuracies):.4f}")
print(f"AUC:      Mean = {np.mean(aucs):.4f}, Std = {np.std(aucs):.4f}")
print(f"TPR:      Mean = {np.mean(tprs):.4f}, Std = {np.std(tprs):.4f}")
print(f"FNR:      Mean = {np.mean(fnrs):.4f}, Std = {np.std(fnrs):.4f}")